In [ ]:
# ==============================================================================
# MỤC 1: SỬ DỤNG THƯ VIỆN (MLXTEND) CHO THUẬT TOÁN APRIORI (KHAI PHÁ LUẬT KẾT HỢP)
# ==============================================================================

# Lưu ý: Nếu máy chưa cài thư viện mlxtend, chạy lệnh: pip install mlxtend
# Import pandas để xử lý dữ liệu dạng bảng (DataFrame)
import pandas as pd
# Import TransactionEncoder để chuyển đổi danh sách giao dịch sang ma trận nhị phân (One-Hot Encoding)
from mlxtend.preprocessing import TransactionEncoder
# Import hàm apriori (tìm tập phổ biến) và association_rules (sinh luật kết hợp) từ thư viện mlxtend
from mlxtend.frequent_patterns import apriori, association_rules

def run_apriori_library(transactions, min_support=0.3, min_confidence=0.7):
    # Bước 1: Mã hóa danh sách các giao dịch (list of lists) thành ma trận True/False
    te = TransactionEncoder()
    te_ary = te.fit(transactions).transform(transactions)
    df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
    
    # Bước 2: Chạy thuật toán Apriori của thư viện mlxtend để tìm các tập mục phổ biến (Frequent Itemsets)
    # use_colnames=True giúp hiển thị tên sản phẩm thay vì chỉ số cột
    frequent_itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)
    
    # Bước 3: Sinh các luật kết hợp (Association Rules) từ tập phổ biến dựa trên ngưỡng độ tin cậy (min_confidence)
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
    
    # Sắp xếp kết quả theo support giảm dần để dễ quan sát và đối chiếu
    frequent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False).reset_index(drop=True)
    if not rules.empty:
        rules = rules.sort_values(by=["confidence", "lift"], ascending=False).reset_index(drop=True)
        
    # Trả về DataFrame tập phổ biến và DataFrame các luật kết hợp
    return frequent_itemsets, rules


In [ ]:
# ==============================================================================
# MỤC 2: TỰ VIẾT CODE THUẬT TOÁN APRIORI TỪ ĐẦU (FROM SCRATCH)
# ==============================================================================

# Import combinations để sinh các tổ hợp tập con và tập ứng viên kích thước k
from itertools import combinations

class CustomApriori:
    def __init__(self, min_support=0.3, min_confidence=0.7):
        # Khởi tạo các ngưỡng siêu tham số cho thuật toán Apriori:
        # min_support: ngưỡng độ hỗ trợ tối thiểu (tỷ lệ giao dịch chứa tập mục)
        # min_confidence: ngưỡng độ tin cậy tối thiểu của luật kết hợp P(Y|X) = support(X U Y) / support(X)
        self.min_support = min_support
        self.min_confidence = min_confidence
        self.frequent_itemsets_ = {}
        self.rules_ = []

    def _get_support(self, itemset, transactions_sets):
        # Tính độ hỗ trợ (support) của một tập mục (frozenset) trên toàn bộ danh sách giao dịch
        count = sum(1 for trans in transactions_sets if itemset.issubset(trans))
        return count / len(transactions_sets)

    def _generate_candidates(self, prev_frequent_itemsets, k):
        # Bước nối (Join step): tạo tập ứng viên C_k có kích thước k từ các tập phổ biến L_{k-1}
        candidates = set()
        prev_list = list(prev_frequent_itemsets)
        for i in range(len(prev_list)):
            for j in range(i + 1, len(prev_list)):
                # Hợp 2 tập phổ biến kích thước k-1 lại với nhau
                union_set = prev_list[i].union(prev_list[j])
                # Chỉ giữ lại nếu hợp của chúng có đúng k phần tử
                if len(union_set) == k:
                    # Bước tỉa (Prune step - Nguyên lý Apriori):
                    # Mọi tập con kích thước k-1 của tập ứng viên C_k đều phải là tập phổ biến
                    all_subsets_frequent = all(
                        frozenset(sub) in prev_frequent_itemsets
                        for sub in combinations(union_set, k - 1)
                    )
                    if all_subsets_frequent:
                        candidates.add(union_set)
        return candidates

    def fit(self, transactions):
        # Chuyển mỗi giao dịch thành kiểu set để kiểm tra tập con (issubset) nhanh hơn
        transactions_sets = [set(t) for t in transactions]
        
        # Bước 1: Tìm tập các mục đơn lẻ duy nhất (C_1) xuất hiện trong dữ liệu
        unique_items = set(item for trans in transactions_sets for item in trans)
        c1 = [frozenset([item]) for item in unique_items]
        
        # Lọc ra tập phổ biến kích thước 1 (L_1) thỏa mãn >= min_support
        current_l = {}
        for itemset in c1:
            sup = self._get_support(itemset, transactions_sets)
            if sup >= self.min_support:
                current_l[itemset] = sup

        # Lưu tất cả các tập phổ biến tìm được vào dictionary self.frequent_itemsets_
        self.frequent_itemsets_ = dict(current_l)
        k = 2

        # Bước 2: Vòng lặp tăng dần kích thước k = 2, 3, ... cho đến khi không còn tập phổ biến mới
        while current_l:
            # Sinh các tập ứng viên C_k từ tập phổ biến L_{k-1}
            candidates_k = self._generate_candidates(current_l.keys(), k)
            next_l = {}
            
            # Tính support cho từng ứng viên C_k và giữ lại các tập đạt ngưỡng min_support -> L_k
            for candidate in candidates_k:
                sup = self._get_support(candidate, transactions_sets)
                if sup >= self.min_support:
                    next_l[candidate] = sup
                    self.frequent_itemsets_[candidate] = sup
            
            current_l = next_l
            k += 1

        # Bước 3: Sinh các luật kết hợp (X -> Y) từ các tập phổ biến có từ 2 phần tử trở lên
        self.rules_ = []
        for itemset, support_xy in self.frequent_itemsets_.items():
            if len(itemset) >= 2:
                # Duyệt qua tất cả các tập con không rỗng X của itemset (với vế phải Y = itemset - X)
                for r in range(1, len(itemset)):
                    for antecedent_tuple in combinations(itemset, r):
                        antecedent = frozenset(antecedent_tuple)
                        consequent = itemset - antecedent
                        
                        support_x = self.frequent_itemsets_[antecedent]
                        support_y = self.frequent_itemsets_[consequent]
                        
                        # Tính độ tin cậy Confidence(X -> Y) = Support(X U Y) / Support(X)
                        confidence = support_xy / support_x
                        
                        # Nếu đạt ngưỡng min_confidence, tính thêm độ nâng Lift và lưu luật
                        if confidence >= self.min_confidence:
                            lift = confidence / support_y
                            self.rules_.append({
                                "antecedents": antecedent,
                                "consequents": consequent,
                                "support": support_xy,
                                "confidence": confidence,
                                "lift": lift
                            })
        return self

    def get_results_dataframes(self):
        # Chuyển đổi kết quả tập phổ biến và luật kết hợp sang DataFrame để dễ so sánh với thư viện
        df_itemsets = pd.DataFrame([
            {"support": sup, "itemsets": itemset}
            for itemset, sup in self.frequent_itemsets_.items()
        ])
        if not df_itemsets.empty:
            df_itemsets = df_itemsets.sort_values(by="support", ascending=False).reset_index(drop=True)

        df_rules = pd.DataFrame(self.rules_)
        if not df_rules.empty:
            df_rules = df_rules.sort_values(by=["confidence", "lift"], ascending=False).reset_index(drop=True)
            
        return df_itemsets, df_rules


In [ ]:
# ==============================================================================
# MỤC 3: TẠO DATA MOCK VÀ KIỂM THỬ SO SÁNH 2 THUẬT TOÁN (THƯ VIỆN VS TỰ VIẾT)
# ==============================================================================

import random

# Bước 1: Tạo dữ liệu giả lập giỏ hàng siêu thị (Mock Transactions Data)
# Cố định hạt giống ngẫu nhiên để tái lập kết quả
random.seed(42)

# Danh mục các sản phẩm phổ biến trong siêu thị
catalog = ["Sữa", "Bánh mì", "Bơ", "Trứng", "Cà phê", "Đường", "Táo", "Chuối"]

# Tạo 200 giao dịch giỏ hàng giả lập có quy luật mua kèm tự nhiên (ví dụ: Bánh mì + Bơ + Sữa, Cà phê + Đường)
mock_transactions = []
for _ in range(200):
    basket = set()
    # Nhóm combo ăn sáng thường đi cùng nhau
    if random.random() < 0.55:
        basket.update(["Bánh mì", "Bơ"])
        if random.random() < 0.75:
            basket.add("Sữa")
    # Nhóm combo đồ uống thường đi cùng nhau
    if random.random() < 0.45:
        basket.add("Cà phê")
        if random.random() < 0.80:
            basket.add("Đường")
    # Thêm ngẫu nhiên 1-2 sản phẩm khác vào giỏ hàng
    extra_items = random.sample(catalog, k=random.randint(1, 2))
    basket.update(extra_items)
    mock_transactions.append(list(basket))

print(f"Tổng số giao dịch giả lập (Mock Transactions): {len(mock_transactions)}")
print("Ví dụ 5 giao dịch đầu tiên:")
for idx in range(5):
    print(f"  Giao dịch {idx+1}: {mock_transactions[idx]}")
print()

# Thiết lập ngưỡng min_support = 0.25 (25%) và min_confidence = 0.70 (70%)
MIN_SUP = 0.25
MIN_CONF = 0.70

# Bước 2: Chạy thử nghiệm với thư viện mlxtend (Mục 1)
df_itemsets_lib, df_rules_lib = run_apriori_library(
    mock_transactions, min_support=MIN_SUP, min_confidence=MIN_CONF
)

# Bước 3: Chạy thử nghiệm với thuật toán tự viết CustomApriori (Mục 2)
custom_apriori = CustomApriori(min_support=MIN_SUP, min_confidence=MIN_CONF)
custom_apriori.fit(mock_transactions)
df_itemsets_custom, df_rules_custom = custom_apriori.get_results_dataframes()

# Bước 4: So sánh kết quả tập phổ biến (Frequent Itemsets) và luật kết hợp (Association Rules)
print("=== KẾT QUẢ ĐÁNH GIÁ & SO SÁNH APRIORI ===")
print(f"Số tập phổ biến tìm được - Thư viện mlxtend : {len(df_itemsets_lib)}")
print(f"Số tập phổ biến tìm được - Thuật toán tự viết: {len(df_itemsets_custom)}")
print(f"Số luật kết hợp sinh ra  - Thư viện mlxtend : {len(df_rules_lib)}")
print(f"Số luật kết hợp sinh ra  - Thuật toán tự viết: {len(df_rules_custom)}\n")

# Kiểm tra độ trùng khớp tuyệt đối về tập phổ biến và luật kết hợp giữa 2 phiên bản
set_itemsets_lib = set(df_itemsets_lib["itemsets"])
set_itemsets_custom = set(df_itemsets_custom["itemsets"])
print(f"Tập phổ biến trùng khớp 100%: {set_itemsets_lib == set_itemsets_custom}")

set_rules_lib = set(zip(df_rules_lib["antecedents"], df_rules_lib["consequents"]))
set_rules_custom = set(zip(df_rules_custom["antecedents"], df_rules_custom["consequents"]))
print(f"Luật kết hợp trùng khớp 100%: {set_rules_lib == set_rules_custom}\n")

# In bảng tập phổ biến (Frequent Itemsets) của thuật toán tự viết
print("Danh sách tập mục phổ biến (Frequent Itemsets - Tự viết):")
df_itemsets_display = df_itemsets_custom.copy()
df_itemsets_display["itemsets"] = df_itemsets_display["itemsets"].apply(lambda x: ", ".join(sorted(list(x))))
print(df_itemsets_display.to_string(index=False))

# In bảng các luật kết hợp (Association Rules) của thuật toán tự viết
print("\nDanh sách luật kết hợp mạnh (Association Rules - Tự viết):")
df_rules_display = df_rules_custom.copy()
df_rules_display["antecedents"] = df_rules_display["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
df_rules_display["consequents"] = df_rules_display["consequents"].apply(lambda x: ", ".join(sorted(list(x))))
print(df_rules_display[["antecedents", "consequents", "support", "confidence", "lift"]].round(4).to_string(index=False))
